# RQ2：不同海洋区域的误差如何随 lead time 增长？

这个 notebook 是 RQ1 六模型误差曲线的区域版本。六个模型仍然使用**全部 training partition**训练，然后把固定的757个test cases按照forecast origin分成Shelf、Kuroshio Core、Transition和Other Offshore。

这里回答的是：**同一个全区域模型在不同海洋环境中的误差增长是否不同？** 它不属于区域重训；区域重训实验位于`../extensions/`。

## 两张核心结果图

### Median separation error：典型case

![Regional median error](../../figures/regional_lead_time/regional_median_error_vs_lead_time.png)

### RMSE：对严重失败case更加敏感

![Regional RMSE](../../figures/regional_lead_time/regional_rmse_vs_lead_time.png)

## 读取24、48、72小时结果

下面的表格不是重新训练结果，而是全区域模型在不同regional test subsets上的表现。

In [ ]:
from pathlib import Path
import pandas as pd
from IPython.display import display

PROJECT_ROOT = next(
    p for p in [Path.cwd(), *Path.cwd().parents]
    if (p / 'RQ2' / 'data' / 'regional_lead_time').exists()
)
RESULTS = PROJECT_ROOT / 'RQ2' / 'data' / 'regional_lead_time'
summary = pd.read_csv(RESULTS / 'six_model_regional_lead_summary.csv')
checkpoints = summary[summary['lead_time_hours'].isin([24, 48, 72])].copy()
display(checkpoints.round(2))

## 怎样解读？

1. **Kuroshio Core的曲线整体最高。** 区域差距不是只在第72小时突然产生，而是随lead time逐渐扩大。
2. **FNN在Kuroshio、Transition和Other Offshore的72-h RMSE最低；KNN在Shelf略优。** 因此不存在一个模型在所有区域都绝对领先。
3. **Kuroshio中的物理基线尤其困难。** Persistence、Mean Flow和VAR(1)随着递归时间增加，与学习模型的差距逐渐扩大。
4. **Median和RMSE的排序不一定完全相同。** RMSE更强调少数大误差案例，median更接近典型case。两者一起看才能判断改善发生在整体还是误差尾部。
5. **这些结果说明区域可预测性不同，但不能独立证明原因。** Kuroshio同时具有更快流速、更长位移和复杂动力过程；后续需要使用标准化误差或控制环境变量。

## 可复现性检查

脚本重新构建了六个模型的逐case、逐小时误差，并将第72小时结果与原RQ2表进行比较。最大数值差异约为1.4×10⁻¹⁴ km，说明新曲线与原来的72小时结果完全一致。

重新运行：

```bash
cd "/Users/roger/Documents/data3001-project"
source .venv/bin/activate
export MPLCONFIGDIR=/tmp/data3001-matplotlib-regional-lead
python RQ2/20_regional_lead_time_curves.py
```